# France Travail — Collecte & export des offres d'emploi
Collecte toutes les offres par département, déduplique, ne garde que les colonnes utiles au dashboard, et exporte en CSV propre.

In [1]:
import requests
import pandas as pd
import time, json
from tqdm.notebook import tqdm

# ── Identifiants API ───────────────────────────────────────────────────────────
CLIENT_ID     = "PAR_jobflow_84188568833432894ecc40a3202abb0451a7505bce481bb92c5203c16647e48e"
CLIENT_SECRET = "d24e0f72412ce7e6e03a36e8c56096524d5b36ce0cb0105765eb15f6e50da31d"

TOKEN_URL  = "https://entreprise.francetravail.fr/connexion/oauth2/access_token?realm=%2Fpartenaire"
SEARCH_URL = "https://api.francetravail.io/partenaire/offresdemploi/v2/offres/search"
SCOPE      = "api_offresdemploiv2 o2dsoffre"
BATCH      = 150      # max par appel API
MAX_RANGE  = 3000     # plafond par département
SLEEP      = 0.12     # ~8 appels/s (limite = 10)
OUTPUT_CSV = "offres_france_travail_clean.csv"

DEPTS = (
    [f"{i:02d}" for i in range(1, 20)]
    + ["2A", "2B"]
    + [f"{i:02d}" for i in range(21, 96)]
    + ["971", "972", "973", "974", "976"]
)

# ── Colonnes à CONSERVER (dashboard-only) ─────────────────────────────────────
# Toutes les autres seront ignorées dès json_normalize → aucun CSV intermédiaire lourd.
COLS_KEEP = [
    # Identifiant & dates
    "id",
    "intitule",
    "dateCreation",
    "dateActualisation",
    # Type de contrat / poste
    "typeContrat",
    "typeContratLibelle",
    "natureContratLibelle",
    "experienceLibelle",
    "niveauExperience",           # si présent
    "qualificationLibelle",
    "dureeTravailLibelleConverti",
    # Lieu
    "lieuTravail.libelle",
    "lieuTravail.codePostal",
    "lieuTravail.departement",
    # Entreprise
    "entreprise.nom",
    "entreprise.secteurActiviteLibelle",
    # Salaire
    "salaire.libelle",
    # Métier / domaine
    "appellationlibelle",
    "secteurActiviteLibelle",
    # Contexte travail
    "contexteTravail.typesTravail",
    # Compétences & formations (listes sérialisées)
    "competences",
    "formations",
    "langues",
    "permis",
    # Description (utile pour recherche plein-texte)
    # Agence
    "agence.libelle",
    "agence.urlPostulation",
    # Offre
    "origineOffre.urlOrigine",     # gardé pour lien direct
]

In [2]:
class TokenManager:
    """Gère le token OAuth2 et le renouvelle automatiquement."""
    def __init__(self):
        self.token = None
        self.expires_at = 0

    def get(self):
        if time.time() < self.expires_at - 60:
            return self.token
        resp = requests.post(
            TOKEN_URL,
            data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
                  "client_secret": CLIENT_SECRET, "scope": SCOPE},
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=15,
        )
        resp.raise_for_status()
        data = resp.json()
        self.token = data["access_token"]
        self.expires_at = time.time() + data.get("expires_in", 1499)
        return self.token

tm = TokenManager()
print(f"✅ Token OK — expire dans {int(tm.expires_at - time.time())}s")

✅ Token OK — expire dans -1779108966s


In [3]:
def fetch_dept(dept_code, token_mgr):
    """Récupère toutes les offres d'un département (pagination complète)."""
    offres, errors = [], 0
    for start in range(0, MAX_RANGE, BATCH):
        headers = {"Authorization": f"Bearer {token_mgr.get()}", "Accept": "application/json"}
        params  = {"departement": dept_code, "range": f"{start}-{start + BATCH - 1}"}
        try:
            resp = requests.get(SEARCH_URL, headers=headers, params=params, timeout=25)
            if resp.status_code in (200, 206):
                resultats = resp.json().get("resultats", [])
                offres.extend(resultats)
                if len(resultats) < BATCH:
                    break
            elif resp.status_code == 204:
                break
            elif resp.status_code == 429:
                time.sleep(2)
                continue
            else:
                errors += 1
                if errors > 3:
                    break
        except requests.RequestException:
            errors += 1
            time.sleep(1)
            if errors > 3:
                break
        time.sleep(SLEEP)
    return offres

In [4]:
all_offres = {}  # id → offre  (déduplication automatique)

for dept in tqdm(DEPTS, desc="Départements", unit="dept"):
    for o in fetch_dept(dept, tm):
        oid = o.get("id")
        if oid and oid not in all_offres:
            all_offres[oid] = o

print(f"\n📦 Collecte terminée : {len(all_offres):,} offres uniques")

Départements:   0%|          | 0/101 [00:00<?, ?dept/s]


📦 Collecte terminée : 266,363 offres uniques


In [7]:
# ── Normalisation ──────────────────────────────────────────────────────────────
df_full = pd.json_normalize(list(all_offres.values()), sep=".")

# Garder uniquement les colonnes utiles présentes dans le DataFrame
cols_present = [c for c in COLS_KEEP if c in df_full.columns]
df = df_full[cols_present].copy()

# Sérialiser les listes (compétences, formations, langues, permis…)
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, list)).any():
        df[col] = df[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
        )

# Convertir les dates
for col in [c for c in df.columns if "date" in c.lower()]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"✅ DataFrame final : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"   Mémoire : {df.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
print("\nColonnes conservées :")
for c in df.columns:
    print(f"  • {c}")

✅ DataFrame final : 266,363 lignes × 20 colonnes
   Mémoire : 393 Mo

Colonnes conservées :
  • id
  • intitule
  • dateCreation
  • dateActualisation
  • typeContrat
  • typeContratLibelle
  • experienceLibelle
  • qualificationLibelle
  • dureeTravailLibelleConverti
  • lieuTravail.libelle
  • lieuTravail.codePostal
  • entreprise.nom
  • salaire.libelle
  • appellationlibelle
  • secteurActiviteLibelle
  • competences
  • formations
  • langues
  • permis
  • origineOffre.urlOrigine


In [6]:
import os

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

size_mb = os.path.getsize(OUTPUT_CSV) / 1024 / 1024
print(f"✅ Export terminé : {OUTPUT_CSV}")
print(f"   {df.shape[0]:,} lignes × {df.shape[1]} colonnes — {size_mb:.1f} Mo")

✅ Export terminé : offres_france_travail_clean.csv
   266,363 lignes × 20 colonnes — 174.7 Mo
